# Импорты

In [1]:
from collections.abc import Callable
from typing import List

import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Математическая модель

Уравнение:<br>
$\displaystyle
\begin{cases}
    \left[
        \begin{array}{l}
                \displaystyle y' = \frac{3y}{x} \\
                \\
                \displaystyle y' = \frac{y}{3x}
        \end{array}
    \right. \\
    y(1) = -1
\end{cases} \\
$
Решение:<br>
$\displaystyle
\left[
    \begin{array}{l}
        y = -x^3 \\ 
        y = -\sqrt[3]{x}
    \end{array}
\right.
$

In [2]:
def f_1(x: float, y: float) -> float:
    return 3 * y / x

def f_2(x: float, y: float) -> float:
    return y / x / 3

def y_1(x: float) -> float:
    return -x ** 3

def y_2(x: float) -> float:
    return -np.cbrt(x)

y_0 = -1

x_0 = 1
x_n = 10

# Решение

Решим уравнения методом Рунге-Кутта 4 порядка:<br>
$\displaystyle
y_{i + 1} = y_i + \frac{h}{6} (k_1 + 2k_2 + 2k_3 + k_4) \\
k_1 = f(x_i, y_i) \\
k_2 = f \left( x_i + \frac{h}{2}, y_i + \frac{h}{2} k_1 \right) \\
k_3 = f \left( x_i + \frac{h}{2}, y_i + \frac{h}{2} k_2 \right) \\
k_4 = f(x_i + h, y_i + h k_3)
$

In [3]:
def runge_kutta_4(n: int, f: Callable[[float, float], float], x: List[float]) -> List[float]:
    h = (x_n - x_0) / n
    y_numeric = [y_0]

    for i in range(n):
        k_1 = f(x[i], y_numeric[i])
        k_2 = f(x[i] + h / 2, y_numeric[i] + h / 2 * k_1)
        k_3 = f(x[i] + h / 2, y_numeric[i] + h / 2 * k_2)
        k_4 = f(x[i] + h, y_numeric[i] + h * k_3)

        y_numeric.append(y_numeric[i] + h / 6 * (k_1 + 2 * k_2 + 2 * k_3 + k_4))

    return y_numeric

# Графики

In [4]:
def compare_solutions(n: int, f: Callable[[float, float], float], y: Callable[[float], float], title: str):
    x = np.linspace(x_0, x_n, n + 1)

    y_exact = np.array([y(xi) for xi in x])
    y_numeric = np.array(runge_kutta_4(n, f, x))
    error = np.abs(y_exact - y_numeric)

    figure = make_subplots(rows=1, cols=2)

    figure.add_trace(
        go.Scatter(x=x, y=y_exact, mode="lines", name="Точное"),
        row=1, col=1,
    )
    figure.add_trace(
        go.Scatter(x=x, y=y_numeric, mode="lines", name="Численное"),
        row=1, col=1,
    )

    figure.add_trace(
        go.Scatter(x=x, y=error, mode="lines", name="Погрешность",
                   showlegend=True),
        row=1, col=2,
    )

    figure.update_yaxes(exponentformat="power", row=1, col=1)
    figure.update_yaxes(exponentformat="power", row=1, col=2)
    figure.update_xaxes(title_text="x", row=1, col=1)
    figure.update_xaxes(title_text="x", row=1, col=2)
    figure.update_yaxes(title_text="y", row=1, col=1)
    figure.update_yaxes(title_text="y", row=1, col=2)

    figure.update_layout(
        height=600,
        legend=dict(
            orientation="v", 
            xanchor="left", 
            yanchor="bottom", 
            x=0,
            y=0
        ),
    )
    figure.show()

ns = [10, 100, 1000]
for n in ns:
    compare_solutions(n, f_1, y_1, "3y = xy'")
    compare_solutions(n, f_2, y_2, "y = 3xy'")